# Week 2 exercise — Technical tutor (Gradio prototype)

The week-1 tech Q&A tool, now a full prototype: **Gradio chat UI**, **streaming**, a
**system prompt** for expertise, and a dropdown to **switch models** across providers
(`gpt-4o-mini`, `claude-haiku-4.5`, local `llama3.2`).

In [1]:
import gradio as gr
import anthropic
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

frontier = OpenAI()
claude = anthropic.Anthropic()
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# label -> (provider, model id)
MODELS = {
    "gpt-4o-mini": ("openai", "gpt-4o-mini"),
    "claude-haiku-4.5": ("anthropic", "claude-haiku-4-5-20251001"),
    "llama3.2": ("ollama", "llama3.2"),
}

SYSTEM = "You are an expert technical tutor. Explain clearly and concisely, with a short example when useful. Use markdown."

In [2]:
def chat(message, history, model_label):
    """Stream a reply. `history` is Gradio's list of {role, content} messages."""
    provider, model = MODELS[model_label]
    if provider == "anthropic":
        with claude.messages.stream(
            model=model, system=SYSTEM, max_tokens=1000,
            messages=history + [{"role": "user", "content": message}],
        ) as stream:
            reply = ""
            for text in stream.text_stream:
                reply += text
                yield reply
    else:
        client = frontier if provider == "openai" else local
        messages = [{"role": "system", "content": SYSTEM}] + history + [{"role": "user", "content": message}]
        stream = client.chat.completions.create(model=model, messages=messages, stream=True)
        reply = ""
        for chunk in stream:
            reply += chunk.choices[0].delta.content or ""
            yield reply

Quick check that each model streams (prints the final reply):

In [3]:
for label in MODELS:
    reply = ""
    for reply in chat("In one sentence, what is a Python decorator?", [], label):
        pass
    print(f"[{label}] {reply}\n")

[gpt-4o-mini] A Python decorator is a design pattern that allows you to modify the behavior of a function or method by wrapping it in another function, typically used to add functionality, like logging or access control, without altering the original function's code.



[claude-haiku-4.5] A decorator is a function that modifies or wraps another function or class to change its behavior without permanently altering its source code.



[llama3.2] A Python decorator is a small function that takes another function as an argument and returns a new function that "wraps" the original function, allowing for the addition of additional functionality or behavior to be applied before or after the execution of the original function.

**Example:**
```python
def my_decorator(func):
    def wrapper():
        print("Before executing the function")
        func()
        print("After executing the function")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()  # Output: Before executing the function, Hello!, After executing the function
```
In this example, `my_decorator` is a decorator that prints a message before and after the execution of the `say_hello()` function. The `@my_decorator` syntax before the function definition allows for easy application of the decorator to existing code.



Launch the UI:

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Technical tutor",
    additional_inputs=[gr.Dropdown(list(MODELS), value="gpt-4o-mini", label="Model")],
)
demo.launch()